# Reading a representation, testing a mechanism
## Two paper replications in mechanistic interpretability

We’ll investigate two questions: **Do sparse autoencoders help us read information from a model? And how can we test which internal computations affect an answer?**

We’ll start with [Are Sparse Autoencoders Useful?](https://arxiv.org/pdf/2502.16681), reproduce its comparison of probing methods, then turn to [Locating and Editing Factual Associations in GPT](https://arxiv.org/pdf/2202.05262) and trace a factual prediction.

Work in pairs, with LLM assistance. For each experiment, make a prediction before running it and explain what the result supports. Aim to reproduce the original result before changing the experiment.


# 1️⃣ Do SAE features make better probes?

Suppose we want to tell whether a model is processing a statement about basketball. We could save its internal vectors—its **activations**—and train a small classifier to predict that label. This classifier is a **probe**. The language model stays fixed; only the probe learns.

A **linear probe** reads the vector by learning a weighted sum of its entries. This can reveal information that is available inside the model, although it doesn’t show whether the model uses that information to produce its answer.

The paper asks whether we can make probing work better by changing the representation the probe reads. Its proposed ingredient is a sparse autoencoder.


## Why might an SAE help?

A model’s activation entries don’t come with names. One entry may respond to several unrelated things, while a useful concept may be spread across many entries.

A **sparse autoencoder (SAE)** learns another way to describe these vectors. Its encoder turns an activation into a larger set of features, most of which are zero for any particular input. Its decoder tries to reconstruct the original activation from those few active features. Training balances accurate reconstruction against using fewer features.

If these features separate useful concepts, a probe might find the relevant signal more easily. But recognizable features alone don’t establish that advantage. We need to compare their performance with other methods.

That gives us the experiment: train probes on the original activations and on SAE features, then test whether adding SAE probes improves the available set of methods. **Predict: when would you expect the SAE features to help?**


## Set up the replication

We’ll reproduce the paper’s **Figure 4**, using Gemma-2-9B, all 113 tasks, and the original candidate methods. First we’ll reconstruct the comparison from the authors’ released scores. Then we’ll rerun the training that produced those scores.

Run the setup cells in a fresh GPU runtime, and **restart the runtime when the install cell finishes** (Colab: *Runtime → Restart session*), then continue from the setup cell. The install swaps some of Colab’s preinstalled packages, such as `transformers`, for the older versions this notebook was tested with. Set `USE_SOLUTIONS=True` for the reference implementations; otherwise complete the exercises as you go. Large data downloads and activation extraction should be prepared before the live session. File locations are explained where we load them below.


In [ ]:
%pip -q install "transformer-lens==2.15.4" "transformers==4.51.3" "datasets==2.21.0" "scikit-learn==1.6.1" "xgboost==2.1.4" "matplotlib==3.10.1" natsort huggingface_hub
# Tested on Python 3.13, Colab's runtime since September 2026. Restart the runtime after this cell.
# - No sae-lens: 5.10.6 pins pyzmq==26.0.0, lxml 4.9 and numpy<2, which have no Python 3.13 builds, and its
#   pyzmq pin would replace the copy Colab's own kernel uses. We read the Gemma Scope SAE parameters directly.
# - Keep transformer-lens at 2.15.4: TransformerLens 4.0 removed HookedTransformer.
# - Using sae-lens 5.10.6 on Python 3.12 instead? Also pin "pyarrow<26": pyarrow 26 will not import with numpy<2.


In [ ]:
#@title Setup and imports
import os, sys, gc, json, random, hashlib, subprocess, inspect, ast, time, warnings
from pathlib import Path
from importlib.metadata import version as installed_version
for module, dist in [('numpy', 'numpy'), ('transformers', 'transformers'), ('huggingface_hub', 'huggingface-hub')]:
    if module in sys.modules and sys.modules[module].__version__ != installed_version(dist):
        raise RuntimeError(f'{module} {sys.modules[module].__version__} is loaded but {installed_version(dist)} is '
                           'installed: restart the runtime (Colab: Runtime → Restart session), then rerun this cell.')
from functools import partial
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from transformer_lens import HookedTransformer

USE_SOLUTIONS = False
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
COMMIT = '9917d94e8ab568e3686b6d3801fab52cda54b81d'
# Colab keeps everything in /content; on a workshop pod, next to this notebook
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
REPO = ROOT / 'SAE-Probing'
RUN_DIR = ROOT / 'workshop_replication'; RUN_DIR.mkdir(exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JoshEngels/SAE-Probes.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Pinned author code:', COMMIT, '| Device:', DEVICE)


## First, understand the comparison we’re trying to reproduce

The paper compares two toolkits: one containing five methods that read the original activations, and another that also includes SAE probes. The five baselines are logistic regression (a linear probe), PCA followed by a probe (first compress the vectors), nearest neighbors (use similar training examples), boosted decision trees, and a small neural network.

For each task, we choose a method using **validation data**: examples reserved for choosing settings. We then report its performance on separate **test data**. Choosing by test performance would reward methods that happened to get lucky on the evaluation.

The next cell loads the authors’ scores and selects the SAE configurations used in this comparison. **Width** is the total number of SAE features; **L0** is the average number active per example; **k** is how many we give the probe.


In [ ]:
baseline_path = REPO / 'results/baseline_probes_gemma-2-9b/normal_settings/layer20_results.csv'
sae_path = REPO / 'results/sae_probes_gemma-2-9b/normal_setting/all_metrics.csv'
released_base = pd.read_csv(baseline_path)
released_sae_all = pd.read_csv(sae_path)

def standard_sae_candidates(frame):
    s = frame[(frame['layer'] == 20) & (frame.reg_type == 'l1') & frame.k.isin([16, 128])].copy()
    s['width'] = s.sae_id.str.extract(r'width_(16k|131k|1m)(?:/|$)')
    s['l0'] = s.sae_id.str.extract(r'l0_(\d+)').astype(int)
    s = s.dropna(subset=['width']).sort_values('l0', ascending=False)
    s = s.drop_duplicates(['layer', 'width', 'dataset', 'k'], keep='first')
    s['method'] = 'sae_' + s.sae_id + '_k' + s.k.astype(str)
    return s

released_sae = standard_sae_candidates(released_sae_all)
TASKS = list(released_base.dataset.drop_duplicates())
SAE_IDS = list(released_sae.sae_id.drop_duplicates())
assert len(TASKS) == 113
assert len(released_base) == 113 * 5 and len(released_sae) == 113 * 6
print('Tasks:', len(TASKS), '| SAE candidates:', SAE_IDS)
display(released_base.head())


### Exercise — choose a method for each task

> **Time:** 7 minutes · **Difficulty:** 🔴🔴⚪⚪⚪

Implement `select_toolkit`: choose the row with the highest validation AUC for each dataset, then retain that method’s test score. Never use test scores to choose the winner.

**AUC** measures how often a classifier ranks a positive example above a negative one. An AUC of 0.5 is chance ranking; 1.0 is perfect. It measures ranking, rather than accuracy at a particular decision threshold.

<details><summary>Hint</summary>Use `groupby('dataset')['val_auc'].idxmax()`, then select those rows and index by dataset name.</details>


In [ ]:
def select_toolkit(scores):
    # Return a DataFrame indexed by dataset, including method, val_auc, test_auc.
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
#@title Reference solution — validation-based selection
def select_toolkit_solution(scores):
    required = {'dataset', 'method', 'val_auc', 'test_auc'}
    assert required <= set(scores.columns)
    assert scores[['val_auc', 'test_auc']].notna().all().all()
    return scores.loc[scores.groupby('dataset')['val_auc'].idxmax()].set_index('dataset').sort_index()
if USE_SOLUTIONS:
    select_toolkit = select_toolkit_solution


In [ ]:
toy_scores = pd.DataFrame({'dataset':['task','task'], 'method':['a','b'],
                          'val_auc':[.8,.7], 'test_auc':[.6,.99]})
assert select_toolkit(toy_scores).loc['task','method'] == 'a', 'Never select on test AUC.'

def summarize_toolkits(base, sae, plot=True, select=None):
    select = select or select_toolkit  # the tie-breaking exercise below passes another rule
    without = select(base)
    with_sae = select(pd.concat([base, sae], ignore_index=True))
    assert without.index.equals(with_sae.index)
    delta = with_sae.test_auc - without.test_auc
    summary = {'tasks':len(delta), 'mean_delta':float(delta.mean()),
               'se_ddof0':float(delta.std(ddof=0) / np.sqrt(len(delta))),
               'sae_selected':int(with_sae.method.str.startswith('sae_').sum())}
    if plot:
        fig, ax = plt.subplots(figsize=(5,5))
        ax.scatter(without.test_auc, with_sae.test_auc, alpha=.5)
        lo = min(without.test_auc.min(), with_sae.test_auc.min())
        hi = max(without.test_auc.max(), with_sae.test_auc.max())
        ax.plot([lo,hi],[lo,hi], 'k--')
        ax.set(xlabel='Test AUC: toolkit without SAE', ylabel='Test AUC: toolkit with SAE',
               title=f"Original standard conditions: {len(delta)} tasks")
        plt.show()
    return summary, without, with_sae

released_summary, released_without, released_with = summarize_toolkits(released_base, released_sae)
print(released_summary)
assert released_summary['sae_selected'] == 16  # The current CSV; the paper's 14 comes from an earlier run (next cells).
assert abs(released_summary['mean_delta'] - (-0.0033205223224021152)) < 1e-10
assert abs(released_summary['se_ddof0'] - 0.0012489546986711895) < 1e-10


## Did adding SAEs help?

Each dot is one task. A point above the diagonal means adding SAE candidates improved the selected method’s test score; below means it made it worse. We average the paired changes across tasks to recover the paper’s comparison.

The released tables give a mean change of about **−0.0033 AUC**, and they select an SAE probe on **16 of 113** tasks. The paper reports 14/113, and its Figure 4 legend reads −0.003 ± 0.002 where these tables give −0.003 ± 0.001. The difference is in the data, not the analysis. The repo ships two versions of the SAE probe results. `all_metrics_old.csv` is the file as committed on 23 February 2025, the day the paper appeared on arXiv. On 26–27 March 2025 the authors changed the training script, including the feature ranking you’ll implement below, re-ran the SAE probes, and saved the results as `all_metrics.csv`. The next cell runs the same selection on both files. Record this in your replication notes; the rest of the notebook uses the current file.


In [ ]:
#@title Provided: the paper's numbers come from an earlier run of the SAE probes
sae_old_path = REPO / 'results/sae_probes_gemma-2-9b/normal_setting/all_metrics_old.csv'
released_sae_old = standard_sae_candidates(pd.read_csv(sae_old_path))
assert len(released_sae_old) == 113 * 6
old_summary, _, old_with = summarize_toolkits(released_base, released_sae_old, plot=False)
versions = pd.DataFrame([old_summary, released_summary],
                        index=['all_metrics_old.csv (Feb 2025, the paper)', 'all_metrics.csv (Mar 2025 re-run)'])
versions['Figure 4 legend format'] = [f"Mean Δ: {s['mean_delta']:.3f} ± {s['se_ddof0']:.3f}"
                                      for s in (old_summary, released_summary)]
display(versions)
print('Tasks whose selected method differs between the two files:',
      int((old_with.method != released_with.method).sum()))
assert old_summary['sae_selected'] == 14  # the paper's "14/113"
assert abs(old_summary['mean_delta'] - (-0.0031226460718501853)) < 1e-10
assert abs(old_summary['se_ddof0'] - 0.0015079965530176358) < 1e-10


## Why do the SAE picks lose on test?

**Discuss first:** how could adding choices improve validation performance but worsen test performance?

`select_toolkit` keeps whichever candidate has the best validation AUC. A score picked *because* it was the largest tends to overestimate how that method will do on new data. This is the **winner’s curse**: the more candidates we compare, the more chances one of them has to look lucky.

SAE probes have a second source of optimism, which you’ll see in the authors’ code below. Their features are ranked using the whole training pool, *including the rows later held out for validation*. Those rows helped choose the features, so they no longer give an independent check. This is a form of **leakage**.

For each task where an SAE probe was selected, the plot compares it with the baseline it displaced. **Predict first:** which of the two drops more from validation to test?


In [ ]:
#@title Provided: the selected SAE probe vs the baseline it displaced
sae_tasks = released_with.index[released_with.method.str.startswith('sae_')]
picked, displaced = released_with.loc[sae_tasks], released_without.loc[sae_tasks]
gap_picked = picked.val_auc - picked.test_auc
gap_displaced = displaced.val_auc - displaced.test_auc
n_lost = int((picked.test_auc < displaced.test_auc).sum())
print(f'Tasks where an SAE probe is selected: {len(sae_tasks)}')
print(f'Mean validation − test AUC: selected SAE probe {gap_picked.mean():.4f}, '
      f'displaced baseline {gap_displaced.mean():.4f}')
print(f'The selected SAE probe has the lower test AUC on {n_lost} of {len(sae_tasks)} tasks.')
assert len(sae_tasks) == 16 and n_lost == 15
assert abs(gap_picked.mean() - 0.0530) < 5e-4 and abs(gap_displaced.mean() - 0.0085) < 5e-4

order = (picked.test_auc - displaced.test_auc).sort_values(ascending=False).index
rows = np.arange(len(order))
fig, axes = plt.subplots(1, 2, figsize=(11, 0.3 * len(order) + 1.9), sharex=True, sharey=True)
for ax, metric, title in [(axes[0], 'val_auc', 'Validation AUC (chooses the method)'),
                          (axes[1], 'test_auc', 'Test AUC (what we report)')]:
    base_auc, sae_auc = displaced.loc[order, metric], picked.loc[order, metric]
    ax.hlines(rows, np.minimum(base_auc, sae_auc), np.maximum(base_auc, sae_auc), color='#c3c2b7', lw=2, zorder=1)
    ax.scatter(base_auc, rows, s=36, color='#2a78d6', zorder=2, label='baseline it displaced')
    ax.scatter(sae_auc, rows, s=36, color='#eb6834', zorder=3, label='selected SAE probe')
    ax.set_title(title, loc='left', fontsize=11)
    ax.set_xlabel('AUC')
    ax.grid(axis='x', color='#e1e0d9', lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_yticks(rows, order, fontsize=8)
axes[0].invert_yaxis()
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper right', ncol=2, frameon=False)
fig.suptitle('Tasks where adding SAE probes changed the choice', x=0.01, ha='left')
plt.tight_layout()
plt.show()


### Exercise — break ties the way the paper describes

> **Time:** 8 minutes · **Difficulty:** 🔴🔴⚪⚪⚪

`select_toolkit` uses `idxmax`, which keeps the *first* row with the best validation AUC. Baselines come first in `pd.concat([base, sae])`, so a tie goes to a baseline. The authors’ released standard-condition code (`plot_normal.ipynb`) behaves the same way.

Ties are common here: on 42 of the 113 tasks the best validation AUC is 1.0, and on 40 tasks two or more candidates share the top score. The paper’s Appendix E.1, written for its data-scarcity experiments, states a different rule: on a tie, prefer the SAE probe, then the smallest SAE width, then the largest k.

“Share the top score” needs care, because AUCs are floating-point numbers. On `132_temp_cat_Event Ordering`, the released CSV stores a validation AUC of `1.0000000000000002` for k-nearest neighbours. An AUC above 1 is impossible: this is rounding error, and seven other candidates score exactly 1.0. On `154_athlete_sport_football`, logistic regression has `0.9963636363636365` and the 1M-width, k=128 SAE probe has `0.9963636363636363`, values that differ only in the last binary digit. pandas 2.2 and 3.0 read those two strings as the same number, but pandas 3.1 reads them exactly. With exact comparisons, the paper’s rule picks an SAE probe on 50 tasks under one version of pandas and on 49 under another. We count candidates within `TIE_TOLERANCE = 1e-9` of a task’s best validation AUC as tied, which gives the same answer everywhere (the counts above use it too).

Implement `select_toolkit_paper` with the paper’s rule. **Predict first:** how many tasks will select an SAE probe, and how much will the mean change in test AUC move?

<details><summary>Hint</summary>Keep the rows whose validation AUC is within `tol` of their task’s best. Parse three sort keys from `method`: whether it is an SAE probe, the width rank (16k &lt; 131k &lt; 1m), and k. Sort the tied rows by those keys and keep the first row for each dataset.</details>


In [ ]:
TIE_TOLERANCE = 1e-9  # validation AUCs this close to a task's best count as tied (see above)

def select_toolkit_paper(scores, tol=TIE_TOLERANCE):
    # Like select_toolkit, but among the candidates within tol of the best validation AUC,
    # prefer SAE probes, then the smallest SAE width, then the largest k.
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
#@title Reference solution — the paper's tie-breaking rule
WIDTH_RANK = {'16k': 0, '131k': 1, '1m': 2}

def select_toolkit_paper_solution(scores, tol=TIE_TOLERANCE):
    s = scores.copy()
    s['is_sae'] = s.method.str.startswith('sae_')
    s['width_rank'] = s.method.str.extract(r'width_(16k|131k|1m)/', expand=False).map(WIDTH_RANK).fillna(len(WIDTH_RANK))
    s['k'] = s.method.str.extract(r'_k(\d+)$', expand=False).astype(float).fillna(0)
    tied = s[s.val_auc >= s.groupby('dataset').val_auc.transform('max') - tol]
    # Multi-column sorts are stable, so a tie between baselines alone is broken as idxmax would break it.
    tied = tied.sort_values(['is_sae', 'width_rank', 'k', 'val_auc'], ascending=[False, True, False, False])
    return tied.drop_duplicates('dataset').drop(columns=['is_sae', 'width_rank', 'k']).set_index('dataset').sort_index()

if USE_SOLUTIONS:
    select_toolkit_paper = select_toolkit_paper_solution


In [ ]:
toy = pd.DataFrame({'dataset': ['t'] * 4,
                    'method': ['logreg', 'sae_layer_20/width_1m/average_l0_193_k128',
                               'sae_layer_20/width_16k/average_l0_408_k16', 'sae_layer_20/width_16k/average_l0_408_k128'],
                    'val_auc': [1.0, 1.0, 1.0, 1.0], 'test_auc': [.9, .8, .7, .6]})
assert select_toolkit_paper(toy).loc['t', 'method'] == 'sae_layer_20/width_16k/average_l0_408_k128'
assert select_toolkit_paper(toy.assign(val_auc=[1.0, .9, .9, .9])).loc['t', 'method'] == 'logreg', 'Ties only.'
near_tie = toy.assign(val_auc=[0.9963636363636365, 0.9963636363636363, 0.5, 0.5])
assert select_toolkit_paper(near_tie).loc['t', 'method'] == 'sae_layer_20/width_1m/average_l0_193_k128', 'Use tol.'

paper_summary, paper_without, paper_with = summarize_toolkits(released_base, released_sae, plot=False,
                                                              select=select_toolkit_paper)
assert paper_without.method.equals(released_without.method)  # the baseline-only toolkit has no SAE to prefer
all_candidates = pd.concat([released_base, released_sae], ignore_index=True)
best_val = all_candidates.groupby('dataset').val_auc.transform('max')
n_tied = int((all_candidates[all_candidates.val_auc >= best_val - TIE_TOLERANCE].groupby('dataset').size() > 1).sum())
n_perfect = int((all_candidates.groupby('dataset').val_auc.max() >= 1 - TIE_TOLERANCE).sum())
print(f'Tasks with a tie for the best validation AUC: {n_tied}; tasks whose best validation AUC is 1.0: {n_perfect}')
n_exact = int(select_toolkit_paper(all_candidates, tol=0).method.str.startswith('sae_').sum())
print(f'Exact comparisons (tol=0) with pandas {pd.__version__}: {n_exact} SAE selections; '
      f'with the tolerance: {paper_summary["sae_selected"]}')
comparison = {}
for name, summary, chosen in [('ties → baseline (released code)', released_summary, released_with),
                              ('ties → SAE (paper, App. E.1)', paper_summary, paper_with)]:
    delta = chosen.test_auc - released_without.test_auc
    comparison[name] = {**summary, 'tasks_hurt': int((delta < 0).sum()), 'tasks_helped': int((delta > 0).sum())}
display(pd.DataFrame.from_dict(comparison, orient='index'))
assert (n_tied, n_perfect) == (40, 42)
assert paper_summary['sae_selected'] == 51
assert abs(paper_summary['mean_delta'] - (-0.0036297979206502914)) < 1e-10
assert abs(paper_summary['se_ddof0'] - 0.0012478251415040927) < 1e-10


The count of SAE selections roughly triples, yet the mean change barely moves. On a tied task, validation cannot tell the SAE probe from the baseline, so the rule picks between them by convention; their test scores then decide whether the swap helps or hurts. A count of “SAE wins” depends on that convention, while the paired mean change is far more stable. **Discuss:** which number belongs in a paper’s abstract, and what should the methods section say about ties?

We’ve reconstructed the analysis from the released scores; next we’ll check whether retraining the methods gives the same picture.


## Load the representations used in the paper

To rerun training, we need the original activations and labels. Download them from the [authors’ data release](https://www.dropbox.com/scl/fo/lvajx9100jsy3h9cvis7q/AIocXXICIwHsz-HsXSekC3Y?rlkey=tq7td61h1fufm01cbdu2oqsb5&dl=0), preserve the directory layout, and set `ARTIFACT_ROOT` below.

These vectors come from layer 20, at the final input token. Keeping the same examples, token positions, and layer matters: otherwise we would be testing different representations.

The next cell checks that every task is present. Missing files must be resolved before running the full comparison.


In [ ]:
# Uncomment if original artifacts are in your Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# On a workshop pod: python tools/get_replication_data.py notebook (see the repo README)
ARTIFACT_ROOT = (Path('/content/drive/MyDrive/SAE-Probing') if Path('/content').exists()
                 else Path(os.environ.get('SAE_ARTIFACTS', '/workspace/sae_artifacts')))
DATA_ROOT = ARTIFACT_ROOT / 'data'
MODEL_NAME, LAYER = 'gemma-2-9b', 20

import utils_data as ud
ud.BASEPATH = str(ARTIFACT_ROOT)  # Explicitly fix the author's relative data-root assumption.
master = pd.read_csv(REPO / 'data/probing_datasets_MASTER.csv')
binary = master[master['Data type'] == 'Binary Classification']
dataset_files = {Path(p).stem: DATA_ROOT / p for p in binary['Dataset save name']}
missing = []
for task in TASKS:
    for path in [dataset_files[task], DATA_ROOT / f'model_activations_{MODEL_NAME}/{task}_blocks.20.hook_resid_post.pt']:
        if not path.exists(): missing.append(str(path))
print('Missing raw-artifact files:', len(missing))
for p in missing[:10]: print(p)
RAW_ARTIFACTS_READY = not missing
# Checkpoint A remains runnable without these assets. Checkpoint B requires all of them.


## How does the paper choose SAE features?

The idea is to find features that behave differently in the two labeled groups. For example, a basketball feature might be more active on basketball examples than on other examples. The probe then learns from a small set of these promising features.

There are two details to preserve from the released code. It rescales each feature before ranking the difference between groups, but fits the probe on the original feature values. It also ranks features using the full training pool before dividing that pool for validation. Test labels remain separate.

Read the three short functions below with your LLM. Explain how the split, feature ranking, and validation work. We’ll reproduce that ordering first; selecting features separately within each validation fold is a useful follow-up.


In [ ]:
# Print source without importing the heavyweight training-script dependencies.
def show_function(filename, name):
    source = (REPO / filename).read_text()
    tree = ast.parse(source)
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == name)
    print(ast.get_source_segment(source, node))

show_function('utils_data.py', 'get_xy_traintest_specify')
show_function('train_sae_probes.py', 'get_sorted_indices_new')
show_function('utils_training.py', 'get_cv')


## Turn model activations into SAE features

We now need the SAE features for exactly the same training and test examples. If they were prepared beforehand, the next cell just checks the files. Otherwise, set `GENERATE_ENCODINGS=True` to apply the original pretrained SAEs to the saved activations.

These are Gemma Scope’s **JumpReLU** SAEs, and we read their released parameters directly. The encoder is two lines. A linear map gives one pre-activation per feature, `pre = x @ W_enc + b_enc`; then each feature is kept only where it clears its own learned threshold, `f = pre * (pre > threshold)`. (The decoder maps features back, `x̂ = f @ W_dec + b_dec`; we don’t need it here.) The 1M-feature SAE’s parameter file is 30 GB, so encoding with it is optional (`ENCODE_1M`).

These features are mostly zero, so we store them in a **sparse matrix**, which records the nonzero values. The next exercise works directly with that representation instead of expanding all the zeros.


In [ ]:
#@title Provided: load or prepare SAE features
from huggingface_hub import hf_hub_download

def encoding_paths(task, sae_id):
    _, width, l0 = sae_id.split('/')
    stem = f'{task}_{LAYER}_{width}_{l0}_'
    root = DATA_ROOT / f'sae_activations_{MODEL_NAME}/normal_setting'
    return {name:root / (stem + name + '.pt') for name in
            ['X_train_sae','X_test_sae','y_train','y_test']}

def original_raw_split(task):
    assert RAW_ARTIFACTS_READY, 'Download the original artifacts listed in preflight.'
    size = len(ud.read_numbered_dataset_df(task))
    return ud.get_xy_traintest(min(size - 100, 1024), task, LAYER, MODEL_NAME)

def load_gemma_scope(sae_id):
    # Gemma Scope's released parameters (google/gemma-scope-9b-pt-res, CC-BY-4.0). Each params.npz holds
    # W_enc, b_enc, threshold, W_dec and b_dec; encoding needs only the first three.
    path = hf_hub_download('google/gemma-scope-9b-pt-res', f'{sae_id}/params.npz')
    with np.load(path) as params:
        return {name: torch.from_numpy(params[name]).to(DEVICE) for name in ['W_enc', 'b_enc', 'threshold']}

def jumprelu_encode(x, sae):
    pre = x @ sae['W_enc'] + sae['b_enc']   # one pre-activation per feature
    return pre * (pre > sae['threshold'])   # keep a feature only where it clears its own threshold

# Three toy features with thresholds 0.5, 0.5 and 2.0: the second input falls below its threshold, and the
# third is positive but below its own, higher threshold.
toy_sae = {'W_enc': torch.eye(3), 'b_enc': torch.zeros(3), 'threshold': torch.tensor([0.5, 0.5, 2.0])}
assert torch.equal(jumprelu_encode(torch.tensor([[1.0, 0.2, 1.0]]), toy_sae), torch.tensor([[1.0, 0.0, 0.0]]))

GENERATE_ENCODINGS = False
ENCODE_1M = False  # The 1M-feature SAE's params.npz is 30 GB; encode with it only on a machine with room for it.
if GENERATE_ENCODINGS:
    assert RAW_ARTIFACTS_READY
    for sae_id in [s for s in SAE_IDS if ENCODE_1M or 'width_1m' not in s]:
        sae = load_gemma_scope(sae_id)
        for task in TASKS:
            paths = encoding_paths(task, sae_id)
            if all(p.exists() for p in paths.values()): continue
            Xtr, ytr, Xte, yte = original_raw_split(task)
            paths['X_train_sae'].parent.mkdir(parents=True, exist_ok=True)
            with torch.inference_mode():
                for key, Xpart in [('X_train_sae',Xtr),('X_test_sae',Xte)]:
                    chunks = []
                    for start in range(0,len(Xpart),16):
                        z = jumprelu_encode(Xpart[start:start+16].to(DEVICE, torch.float32), sae).cpu().to_sparse().coalesce()
                        chunks.append(z)
                    torch.save(torch.cat(chunks, dim=0).coalesce(), paths[key])
            torch.save(torch.as_tensor(ytr), paths['y_train'])
            torch.save(torch.as_tensor(yte), paths['y_test'])
        del sae
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

missing_sae = [str(p) for task in TASKS for sid in SAE_IDS
               for p in encoding_paths(task,sid).values() if not p.exists()]
print('Missing split-SAE files:',len(missing_sae))
for p in missing_sae[:5]: print(p)


### Exercise — find the features that distinguish the labels

> **Time:** 10 minutes · **Difficulty:** 🔴🔴🔴⚪⚪

Implement `rank_latents` to match the author function you just read: divide each feature by its average nonzero activation, compare its mean in the two labeled groups, and rank the absolute differences.

This determines what information the SAE probe gets to read. The check below compares your ranking with the original implementation before we use it in the full experiment.

<details><summary>Hint</summary>For sparse inputs, compute column sums, nonzero counts, and class-specific sums from the stored entries. You can rescale the difference afterward without constructing the full normalized matrix.</details>


In [ ]:
def rank_latents(Z, labels):
    # Return all column indices sorted by the original-code score, descending.
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
#@title Reference solution — memory-conscious original ranking
def rank_latents_solution(Z, labels):
    Z = Z.float().cpu()
    y = torch.as_tensor(labels).cpu()
    assert set(y.unique().tolist()) == {0,1}
    if Z.layout == torch.strided:
        nonzero_mean = Z.sum(0) / ((Z != 0).sum(0) + 1e-6)
        normalized = Z / (nonzero_mean + 1e-6)
        score = (normalized[y == 1].mean(0) - normalized[y == 0].mean(0)).abs()
    else:
        Z = Z.to_sparse_coo().coalesce()
        row, col = Z.indices(); value = Z.values()
        keep = value != 0
        row, col, value = row[keep], col[keep], value[keep]
        width = Z.shape[1]
        total = torch.zeros(width); count = torch.zeros(width)
        pos = torch.zeros(width); neg = torch.zeros(width)
        total.scatter_add_(0, col, value)
        count.scatter_add_(0, col, torch.ones_like(value))
        positive = y[row] == 1
        pos.scatter_add_(0, col[positive], value[positive])
        neg.scatter_add_(0, col[~positive], value[~positive])
        nonzero_mean = total / (count + 1e-6)
        score = ((pos / (y == 1).sum() - neg / (y == 0).sum()) / (nonzero_mean + 1e-6)).abs()
    return torch.argsort(score, descending=True)

def selected_columns(Z, indices):
    indices = torch.as_tensor(indices).long().cpu()
    if Z.layout == torch.strided:
        return Z[:,indices].float().cpu().numpy()
    Z = Z.to_sparse_coo().coalesce().cpu()
    row,col = Z.indices(); values = Z.values()
    lookup = torch.full((Z.shape[1],),-1,dtype=torch.long)
    lookup[indices] = torch.arange(len(indices))
    keep = lookup[col] >= 0
    out = torch.zeros((Z.shape[0],len(indices)))
    out.index_put_((row[keep],lookup[col[keep]]),values[keep].float(),accumulate=True)
    return out.numpy()

if USE_SOLUTIONS:
    rank_latents = rank_latents_solution


In [ ]:
gen = torch.Generator().manual_seed(4)
zcheck = torch.rand(12,17,generator=gen)
zcheck[zcheck < .7] = 0
ycheck = torch.tensor([0,1]*6)
# Execute only the original ranking function for a direct implementation check.
src = (REPO/'train_sae_probes.py').read_text()
tree = ast.parse(src)
node = next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='get_sorted_indices_new')
original_ns = {'torch':torch}
exec(compile(ast.Module(body=[node],type_ignores=[]),'author_rank','exec'),original_ns)
expected = original_ns['get_sorted_indices_new'](zcheck,ycheck)
assert torch.equal(rank_latents(zcheck,ycheck),expected)
assert torch.equal(rank_latents(zcheck.to_sparse(),ycheck),expected)
assert np.allclose(selected_columns(zcheck.to_sparse(),expected[:4]),zcheck[:,expected[:4]].numpy())
print('Dense and sparse ranking match the original function on the fixture.')


## Train the candidates

We have the original activations, the matching SAE features, and the rule for selecting features. Now we can fit the probes.

The fitting functions try several settings and choose them using validation performance. One such setting is **regularization**, a penalty that discourages overly large weights and can reduce overfitting. SAE probes use an L1 penalty, which can set weights to zero. The original-activation methods keep the authors’ own fitting procedures.

The provided runner trains all five baselines and six SAE candidates for every task, saving each result so it can resume. Set `RUN_TRAINING=True` once the data checks pass. **Before running, trace one task through the code: which examples choose the features, fit the probe, choose its settings, and evaluate it?**


### Paper vs code: three details to know before you compare

- **The MLP baseline is not tuned.** `find_best_mlp` samples `n_iter=1` configuration after `np.random.seed(42)`, which always gives hidden layers (16, 16, 16), learning rate 3.16e-3 and L2 penalty `alpha` 1e-2. The “MLP” candidate is one fixed network, not a hyperparameter search.
- **Validation is one split at the end of the training pool.** For pools of 128 or more examples, `get_cv` holds out the last min(20%, 100) training rows; smaller pools use cross-validation. Each method’s `val_auc` is its best score over the settings it tried on those rows.
- **The XGBoost search is unseeded.** `find_best_xgboost` draws 10 random configurations with Python’s `random` module, which the released code never seeds, so the released XGBoost rows cannot be reproduced exactly. The runner below seeds `random` and `np.random` for each task and method: your reruns repeat, but they won’t match the authors’ draws.


In [ ]:
#@title Provided: full training runner
import utils_training as ut
METHODS = {'logreg':ut.find_best_reg,'pca':ut.find_best_pcareg,'knn':ut.find_best_knn,
           'xgboost':ut.find_best_xgboost,'mlp':ut.find_best_mlp}
OUT = RUN_DIR/'retrained'; OUT.mkdir(exist_ok=True)

def as_numpy(x):
    return x.detach().cpu().numpy() if isinstance(x,torch.Tensor) else np.asarray(x)

def row_path(task, method):
    key = hashlib.sha256((task+'|'+method).encode()).hexdigest()[:24]
    return OUT/(key+'.json')

def fit_record(task,method,fn,Xtr,ytr,Xte,yte,**kwargs):
    path = row_path(task,method)
    if path.exists(): return json.loads(path.read_text())
    stable_seed = int(hashlib.sha256((task+'|'+method).encode()).hexdigest()[:8],16)
    np.random.seed(stable_seed); random.seed(stable_seed)
    started = time.time()
    metrics = fn(Xtr,ytr,Xte,yte,**kwargs)
    row = dict(dataset=task,method=method,commit=COMMIT,seed=stable_seed,
               seconds=time.time()-started,**{k:float(v) for k,v in metrics.items()})
    path.write_text(json.dumps(row,indent=2))
    return row

def run_original_standard():
    assert RAW_ARTIFACTS_READY and not missing_sae, 'Complete original-artifact preflight first.'
    rows = []
    for ti,task in enumerate(TASKS):
        Xtr,ytr,Xte,yte = map(as_numpy,original_raw_split(task))
        for method,fn in METHODS.items():
            rows.append(fit_record(task,method,fn,Xtr,ytr,Xte,yte))
        for sid in SAE_IDS:
            paths = encoding_paths(task,sid)
            saved_ytr = torch.load(paths['y_train'],map_location='cpu',weights_only=True)
            saved_yte = torch.load(paths['y_test'],map_location='cpu',weights_only=True)
            assert np.array_equal(as_numpy(saved_ytr),ytr), (task,sid,'training-row mismatch')
            assert np.array_equal(as_numpy(saved_yte),yte), (task,sid,'test-row mismatch')
            Ztr = torch.load(paths['X_train_sae'],map_location='cpu',weights_only=True)
            Zte = torch.load(paths['X_test_sae'],map_location='cpu',weights_only=True)
            selected = rank_latents(Ztr,ytr)[:128]
            Atr,Ate = selected_columns(Ztr,selected),selected_columns(Zte,selected)
            for k in [16,128]:
                method = f'sae_{sid}_k{k}'
                rows.append(fit_record(task,method,ut.find_best_reg,
                            Atr[:,:k],ytr,Ate[:,:k],yte,penalty='l1'))
            del Ztr,Zte,Atr,Ate
            gc.collect()
        print(f'Finished {ti+1}/{len(TASKS)}:',task)
    frame = pd.DataFrame(rows)
    frame.to_csv(RUN_DIR/'retrained_standard.csv',index=False)
    return frame

# Full original-model benchmark, not a reduced substitute. Enable after assets pass preflight.
RUN_TRAINING = False
if RUN_TRAINING:
    retrained = run_original_standard()
else:
    print('Checkpoint B has not run. Set RUN_TRAINING=True after original artifacts are ready.')


## Compare your run with the released result

A complete run has **113 tasks × 11 candidates**. First check that nothing is missing. Then compare each candidate’s validation and test AUC with the released scores, and reconstruct the same scatter plot using only your newly trained results.

If the plots differ, investigate one candidate at a time: example order, feature ranking, validation choices, and numerical fitting can all matter. The released code does not record every historical random choice, so exact numerical agreement may require investigation. A matching overall sign is useful evidence, but it doesn’t explain candidate-level discrepancies.


In [ ]:
released_candidates = pd.concat([released_base,released_sae],ignore_index=True)
reference_keys = set(zip(released_candidates.dataset,released_candidates.method))
manifest = dict(commit=COMMIT,target='Figure 4 standard conditions',tasks=len(TASKS),
                model=MODEL_NAME,layer=LAYER,sae_ids=SAE_IDS,ks=[16,128],
                baseline_methods=list(METHODS),training_executed=RUN_TRAINING,
                discrepancy=('Paper (arXiv v1, 2025-02-23): 14/113, -0.003 ± 0.002, reproduced by all_metrics_old.csv; '
                             'current all_metrics.csv (re-run 2025-03-27): 16/113, -0.0033 ± 0.0012'))
manifest['versions'] = {name:__import__('importlib.metadata',fromlist=['version']).version(name)
                        for name in ['torch','scikit-learn','numpy','pandas','xgboost','transformer-lens']}
(RUN_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))

if RUN_TRAINING:
    assert len(retrained)==113*11
    assert not retrained.duplicated(['dataset','method']).any()
    assert set(zip(retrained.dataset,retrained.method))==reference_keys
    joined = retrained.merge(released_candidates[['dataset','method','val_auc','test_auc']],
                            on=['dataset','method'],suffixes=('_new','_released'),validate='one_to_one')
    for metric in ['val_auc','test_auc']:
        joined[metric+'_difference']=joined[metric+'_new']-joined[metric+'_released']
    display(joined.groupby('method')[['val_auc_difference','test_auc_difference']].agg(['mean','max','min']))
    fresh_summary,_,_ = summarize_toolkits(retrained[~retrained.method.str.startswith('sae_')],
                                          retrained[retrained.method.str.startswith('sae_')])
    print('Released:',released_summary,'\nRetrained:',fresh_summary)
    joined.to_csv(RUN_DIR/'candidate_discrepancies.csv',index=False)


## What can we conclude?

Write down whether your run supports the paper’s finding for standard conditions, and which differences remain unexplained. This result concerns the evaluated probing tasks; it does not establish that SAEs cannot help with other interpretability problems.

To reproduce the other conditions, continue with the authors’ scarcity, imbalance, label-noise, and distribution-shift experiments. Their validation procedures differ, so inspect each one rather than reusing the standard-condition analysis unchanged.

Next we’ll ask a different question. A successful probe tells us what we can read from a model. Can we identify an internal computation that affects its answer?


# 2️⃣ Where does a factual prediction come from?

The ROME paper studies how models retrieve associations such as “Eiffel Tower → located in → Paris.” Before trying to edit an association, the authors ask which internal computations help produce it.

Their experiment is **causal tracing**. Run a prompt normally, disrupt the subject’s representation, then restore selected internal values from the normal run. If restoring a value brings back the answer, we’ve learned something about how that computation can recover the prediction.

Replacing an activation this way is called **activation patching**. Unlike a probe, it changes the model’s computation. We’ll implement the tracing experiment first; the paper’s later weight-editing method is an extension.


In [ ]:
# Release encoding/training temporaries before loading GPT-2 XL.
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

TRACE_MODEL = 'gpt2-xl'  # Original-paper architecture; large initial download.
# 'gpt2-small' is a faster method adaptation, but may not know the chosen facts.
# The float32 load peaks at ~13.5 GB of CPU RAM, more than free Colab's ~12.7 GB, and crashes the runtime.
# LOW_RAM loads float16 weights instead: peak ~10 GB of RAM and ~3.4 GB of GPU memory; the causal
# traces below match float32 to within ~0.003. (TransformerLens may warn about float16 weight processing.)
import psutil
LOW_RAM = psutil.virtual_memory().total < 16e9  # or set True / False yourself
TRACE_DTYPE = torch.float16 if LOW_RAM else torch.float32
trace_model = HookedTransformer.from_pretrained(TRACE_MODEL, device=DEVICE, dtype=TRACE_DTYPE)
trace_model.eval()
print(trace_model.cfg.n_layers, 'layers;', trace_model.cfg.d_model, 'residual dimensions;', TRACE_DTYPE)


## Choose a fact the model knows

We can’t investigate retrieval of “Paris” if the model barely predicts it to begin with. Check the candidate prompts below and choose one with a meaningful probability for the expected answer. A natural-sounding prompt is no guarantee: GPT-2 XL continues “The capital of France is” with “ the”, and gives “ Paris” only about 9%. The other candidates are drawn from ROME’s 1,209 *known facts*, prompts that GPT-2 XL completes correctly. Set `PAIR_SEED` to draw a different sample for your pair.

Text is split into **tokens**, which may be words or parts of words. We’ll score a single next token and identify which tokens belong to the subject. The prompt is processed by transformer layers: attention can move information between positions, while MLPs transform each position’s vector. Their contributions accumulate in the **residual stream**, which we’ll patch.


In [ ]:
import urllib.request
# ROME's 1,209 facts that GPT-2 XL completes correctly, used for the paper's causal traces.
KNOWNS_URL = 'https://rome.baulab.info/data/dsets/known_1000.json'
knowns_path = RUN_DIR / 'known_1000.json'
if not knowns_path.exists():
    urllib.request.urlretrieve(KNOWNS_URL, knowns_path)
knowns = json.loads(knowns_path.read_text())
assert len(knowns) == 1209

PAIR_SEED = 0  # Pick your own number to draw a different sample of known facts.
FACTS = [
    ('The Eiffel Tower is located in the city of', 'Eiffel Tower', ' Paris'),
    ('The capital of France is', 'France', ' Paris'),  # natural, but the top token is ' the'
] + [(k['prompt'], k['subject'], ' ' + k['attribute']) for k in random.Random(PAIR_SEED).sample(knowns, 4)]

with torch.inference_mode():
    for prompt, subject, answer in FACTS:
        ids = trace_model.to_tokens(prompt, prepend_bos=False)
        target_ids = trace_model.tokenizer.encode(answer, add_special_tokens=False)
        logits = trace_model(ids)[0, -1].float()
        target = target_ids[0]
        print(prompt, '| target:', repr(answer), '| tokens:', target_ids,
              '| P(first token):', round(logits.softmax(-1)[target].item(), 4),
              '| top:', repr(trace_model.tokenizer.decode([int(logits.argmax())])))

PROMPT, SUBJECT, ANSWER = FACTS[0]  # Change after inspecting the baseline.
tok = trace_model.tokenizer
encoded = tok(PROMPT, add_special_tokens=False, return_offsets_mapping=True)
tokens = torch.tensor([encoded['input_ids']], device=DEVICE)
answer_ids = tok.encode(ANSWER, add_special_tokens=False)
assert len(answer_ids) == 1, 'Choose a single-token answer for this core exercise.'
answer_id = answer_ids[0]
start = PROMPT.index(SUBJECT); end = start + len(SUBJECT)
subject_positions = [i for i, (a, b) in enumerate(encoded['offset_mapping'])
                     if b > start and a < end]
assert subject_positions
token_labels = [tok.decode([int(i)]) for i in tokens[0]]
display(pd.DataFrame({'position': range(tokens.shape[1]), 'token': token_labels,
                      'subject?': [i in subject_positions for i in range(tokens.shape[1])]}))


## Disrupt the subject

We add random noise to the subject’s token embeddings—the vectors that first represent its tokens. Does the answer become less likely? That drop gives us something to try to recover.

We follow ROME’s recipe. The noise is Gaussian with standard deviation 3σ, where σ is the standard deviation of the token embeddings of all 1,209 known subjects, each tokenized on its own. We draw 10 noise samples from `np.random.RandomState(1)`, as ROME does.

Run several noise samples, but reuse those exact samples for every patch location. Otherwise, a location might look better just because its corruption happened to be weaker.

If the answer probability barely drops, choose another prompt or examine the noise strength before interpreting a recovery map.


In [ ]:
N_NOISE = 10  # ROME's number of noise samples
NOISE_MULTIPLIER = 3.0
NOISE_SEED = 1  # ROME draws its noise from np.random.RandomState(1)
trace_layers = list(range(0, trace_model.cfg.n_layers, 4))
if trace_model.cfg.n_layers - 1 not in trace_layers:
    trace_layers.append(trace_model.cfg.n_layers - 1)
trace_positions = list(range(tokens.shape[1]))
hook_names = [f'blocks.{layer}.hook_resid_post' for layer in trace_layers]

# ROME's noise level (collect_embedding_std): sigma is the standard deviation of the token embeddings of
# every known subject, each tokenized on its own without a BOS token. (TransformerLens centres each
# embedding row, which moves sigma by less than 0.01% from ROME's value on the raw embeddings.)
subject_token_ids = torch.tensor([i for k in knowns for i in tok.encode(k['subject'], add_special_tokens=False)],
                                 device=DEVICE)
with torch.inference_mode():
    clean_logits, clean_cache = trace_model.run_with_cache(tokens, names_filter=hook_names)
    clean_prob = clean_logits[0, -1].float().softmax(-1)[answer_id].item()
    noise_scale = NOISE_MULTIPLIER * trace_model.W_E[subject_token_ids].float().std().item()
print(f'Noise: {N_NOISE} samples of scale {noise_scale:.4f} (3σ over {len(subject_token_ids):,} subject tokens)')
if TRACE_MODEL == 'gpt2-xl':
    assert abs(noise_scale - 0.1346) < 5e-4, noise_scale

noise_draws = np.random.RandomState(NOISE_SEED).randn(N_NOISE, len(subject_positions), trace_model.cfg.d_model)
fixed_noise = torch.from_numpy(noise_draws).float().to(DEVICE) * noise_scale
repeated_tokens = tokens.repeat(N_NOISE, 1)

def corrupt_subject(activation, hook):
    out = activation.clone()
    out[:, subject_positions, :] += fixed_noise.to(out.dtype)
    return out

with torch.inference_mode():
    corrupt_logits = trace_model.run_with_hooks(repeated_tokens,
                                fwd_hooks=[('hook_embed', corrupt_subject)])
    corrupt_probs = corrupt_logits[:, -1].float().softmax(-1)[:, answer_id].cpu().numpy()
corrupt_prob = float(corrupt_probs.mean())
gap = clean_prob - corrupt_prob
print(f'Clean P={clean_prob:.4f}; corrupted P={corrupt_prob:.4f}; gap={gap:.4f}')
if gap <= 0.01:
    warnings.warn('Weak or reversed corruption effect. Inspect another prompt/scale before interpreting normalized recovery.')


### Exercise — restore one clean state

> **Time:** 8 minutes · **Difficulty:** 🔴🔴⚪⚪⚪

Implement `restore_state`. At the selected token position, replace every corrupted run’s vector with the clean run’s vector. Leave other positions unchanged and avoid modifying the input in place.

This is the central intervention: can one normal internal state rescue an otherwise disrupted prediction?

<details><summary>Hint</summary>Clone the input, then assign `clean_state[0, position, :]` to `out[:, position, :]`. PyTorch repeats the clean vector across the runs.</details>


In [ ]:
def restore_state(activation, hook, clean_state, position):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
#@title Reference solution — activation restoration
def restore_state_solution(activation, hook, clean_state, position):
    out = activation.clone()
    out[:, position, :] = clean_state[0, position, :].to(out.device, out.dtype)
    return out

if USE_SOLUTIONS:
    restore_state = restore_state_solution


In [ ]:
a = torch.zeros(3, 4, 5); c = torch.arange(20.).reshape(1, 4, 5)
b = restore_state(a, None, c, 2)
assert torch.equal(b[:, 2], c[:, 2].expand(3, -1))
assert torch.equal(b[:, [0, 1, 3]], a[:, [0, 1, 3]])
assert torch.equal(a, torch.zeros_like(a)), 'Do not mutate the input.'
print('Restoration checks passed.')


## Find where restoration helps

Apply your hook at each layer and token position. The hook is simply a callback that replaces a value during the forward pass. At each location, measure the change in answer probability relative to the same corrupted runs:

$$\text{restoration effect}=P(\text{answer after patch})-P(\text{answer when corrupted}).$$

The heatmap shows these effects. Predict where you expect the strongest recovery before running it.

We also compute recovery divided by the clean–corrupted probability gap. This normalized score is useful only when that gap is substantial; it can be negative or exceed one. It is not the fraction of a fact “stored” at that location.


In [ ]:
restored_by_replica = np.empty((len(trace_layers), len(trace_positions), N_NOISE))
with torch.inference_mode():
    for li, layer in enumerate(trace_layers):
        name = f'blocks.{layer}.hook_resid_post'
        for pi, position in enumerate(trace_positions):
            patch = partial(restore_state, clean_state=clean_cache[name], position=position)
            logits = trace_model.run_with_hooks(repeated_tokens,
                fwd_hooks=[('hook_embed', corrupt_subject), (name, patch)])
            restored_by_replica[li, pi] = logits[:, -1].float().softmax(-1)[:, answer_id].cpu().numpy()
        print('Finished layer', layer)

paired_effects = restored_by_replica - corrupt_probs[None, None, :]
effects = paired_effects.mean(-1)
recovery = effects / gap if gap > 0.01 else np.full_like(effects, np.nan)

fig, ax = plt.subplots(figsize=(max(8, len(token_labels) * .55), 5))
image = ax.imshow(effects, aspect='auto', cmap='RdBu_r',
                  vmin=-max(abs(effects).max(), 1e-6), vmax=max(abs(effects).max(), 1e-6))
ax.set_xticks(range(len(token_labels)), [repr(t) for t in token_labels], rotation=60, ha='right')
ax.set_yticks(range(len(trace_layers)), trace_layers)
ax.set(xlabel='Restored token position', ylabel='Layer (zero-indexed)',
       title=f'Causal restoration: {PROMPT!r} → {ANSWER!r}')
fig.colorbar(image, ax=ax, label='Change in answer probability vs same corrupted replicas')
plt.tight_layout(); plt.show()


## Check that the patch does what we think

Two simple controls help catch mistakes. Patching a clean state back into a clean run should change nothing. Restoring every token at a residual-stream boundary should recover the clean downstream prediction.

Run both checks before interpreting the map. If they pass, choose a bright region and explain what its restoration effect establishes—and what it leaves open.


In [ ]:
control_name = hook_names[len(hook_names) // 2]
control_position = subject_positions[-1]
with torch.inference_mode():
    identity = trace_model.run_with_hooks(tokens, fwd_hooks=[(control_name,
        partial(restore_state, clean_state=clean_cache[control_name], position=control_position))])
    identity_prob = identity[0, -1].float().softmax(-1)[answer_id].item()

    def restore_all(activation, hook):
        return clean_cache[control_name].expand_as(activation).clone()

    full = trace_model.run_with_hooks(repeated_tokens,
        fwd_hooks=[('hook_embed', corrupt_subject), (control_name, restore_all)])
    full_probs = full[:, -1].float().softmax(-1)[:, answer_id]

assert abs(identity_prob - clean_prob) < 1e-4
assert torch.allclose(full_probs, torch.full_like(full_probs, clean_prob), atol=1e-4, rtol=1e-3)
print('Identity and full-restoration controls passed.')


## Trace the MLP and attention outputs, as in the paper

Restoring the residual stream at a layer restores everything computed up to that point. ROME separates the two kinds of sublayer by restoring only the **MLP outputs** or only the **attention outputs**. One sublayer’s output usually has a small effect on its own, so ROME restores a **window of 10 consecutive layers** around each layer *l*: layers *l*−5 to *l*+4, cut off at the first and last layer.

The next cell computes all three maps (hidden states, MLP, attention) for all 48 layers and every token, using the same 10 noise samples. Instead of one forward pass per (layer, token), it stacks the corrupted runs for every token position into one batch: rows *p*·10 to *p*·10+9 restore token *p*. That gives the same numbers as the loop above, about ten times faster. The cell checks this against your hidden-state map.

**Predict:** at which tokens and layers will restoring the MLP outputs matter most? Where will attention matter?


In [ ]:
#@title Provided: batched causal traces for hidden states, MLP outputs, and attention outputs
WINDOW = 10
TRACE_HOOKS = {'hidden': 'blocks.{}.hook_resid_post', 'mlp': 'blocks.{}.hook_mlp_out', 'attn': 'blocks.{}.hook_attn_out'}
n_layers, n_tokens = trace_model.cfg.n_layers, tokens.shape[1]

def window_layers(kind, layer):
    # ROME's convention: one layer for hidden states, else range(max(0, l - 5), min(L, l + 5)).
    if kind == 'hidden':
        return [layer]
    return list(range(max(0, layer - WINDOW // 2), min(n_layers, layer - (-WINDOW // 2))))

with torch.inference_mode():
    _, full_cache = trace_model.run_with_cache(
        tokens, names_filter=lambda name: name.endswith(('hook_resid_post', 'hook_mlp_out', 'hook_attn_out')))

batch_tokens = tokens.repeat(n_tokens * N_NOISE, 1)
batch_noise = fixed_noise.repeat(n_tokens, 1, 1)         # row p*N_NOISE + j gets noise sample j ...
batch_rows = torch.arange(n_tokens * N_NOISE, device=DEVICE)
batch_positions = batch_rows // N_NOISE                  # ... and restores token p

def corrupt_batch(activation, hook):
    out = activation.clone()
    out[:, subject_positions, :] += batch_noise.to(out.dtype)
    return out

def restore_batch(activation, hook):
    out = activation.clone()
    out[batch_rows, batch_positions] = full_cache[hook.name][0, batch_positions].to(out.dtype)
    return out

traces = {}
started = time.time()
with torch.inference_mode():
    for kind, hook_template in TRACE_HOOKS.items():
        traces[kind] = np.zeros((n_layers, n_tokens))
        for layer in range(n_layers):
            hooks = [('hook_embed', corrupt_batch)]
            hooks += [(hook_template.format(l), restore_batch) for l in window_layers(kind, layer)]
            logits = trace_model.run_with_hooks(batch_tokens, fwd_hooks=hooks)
            probs = logits[:, -1].float().softmax(-1)[:, answer_id].view(n_tokens, N_NOISE).mean(-1)
            traces[kind][layer] = probs.cpu().numpy() - corrupt_prob
print(f'{len(TRACE_HOOKS) * n_layers} forward passes of {n_tokens * N_NOISE} sequences in {time.time() - started:.0f} s')

# The batched hidden-state trace must reproduce the one-position-at-a-time loop above.
loop_difference = np.abs(traces['hidden'][trace_layers][:, trace_positions] - effects).max()
print(f'Largest difference from the loop above: {loop_difference:.1e}')
assert loop_difference < (1e-4 if TRACE_DTYPE == torch.float32 else 5e-3)


In [ ]:
TRACE_STYLE = {'hidden': ('Purples', 'Restore hidden states', 'single restored layer'),
               'mlp': ('Greens', 'Restore MLP outputs', f'center of {WINDOW} restored MLP layers'),
               'attn': ('Reds', 'Restore attention outputs', f'center of {WINDOW} restored attention layers')}
starred = [repr(t) + (' *' if i in subject_positions else '') for i, t in enumerate(token_labels)]
fig, axes = plt.subplots(1, 3, figsize=(17, 0.33 * n_tokens + 2.2), sharey=True)
for ax, (kind, (cmap, title, xlabel)) in zip(axes, TRACE_STYLE.items()):
    # Like ROME's plots, the color scale starts at the corrupted run; negative effects show as white.
    image = ax.imshow(traces[kind].T, aspect='auto', cmap=cmap, vmin=0, vmax=max(traces[kind].max(), 1e-6),
                      interpolation='nearest')
    ax.set_title(title, loc='left', fontsize=11)
    ax.set_xlabel(f'{xlabel} (of {n_layers})')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.02, label='change in P(answer) vs corrupted')
axes[0].set_yticks(range(n_tokens), starred)
fig.suptitle(f'Causal traces for {PROMPT!r} → {ANSWER!r}   (* = subject token)', x=0.01, ha='left')
plt.tight_layout()
plt.show()

peaks = {}
for kind, trace in traces.items():
    layer, position = np.unravel_index(trace.argmax(), trace.shape)
    peaks[kind] = (round(float(trace.max()), 3), int(layer), token_labels[position])
    print(f'{kind:>6}: largest effect {trace.max():.3f} at layer {layer}, token {token_labels[position]!r}')
if TRACE_MODEL == 'gpt2-xl' and PROMPT == 'The Eiffel Tower is located in the city of':
    # Reference values for the default prompt; float16 and float32 agree to within ~0.003.
    for kind, (value, layer, token) in {'mlp': (0.48, 14, 'el'), 'attn': (0.34, 27, ' of')}.items():
        assert peaks[kind][1:] == (layer, token) and abs(peaks[kind][0] - value) < 0.02, (kind, peaks[kind])


Compare these maps with Figure 1 of the paper. Which subject token carries the strongest MLP effect, and at which layers? Where does attention matter instead? The paper places the early MLP effect at the *last* subject token. Is that where it is for this prompt, and what would you check before calling it a real difference?

Remember what a window measures: each MLP cell restores 10 layers at once, so neighbouring cells share most of their layers. A peak at layer *l* says that restoring the MLP outputs of layers *l*−5 to *l*+4 at that token brings the answer back. It does not say that one layer stores the fact.


## From tracing to editing

A successful patch shows that a clean state can restore an answer under this corruption. It doesn’t establish that the location uniquely stores the fact, or that editing weights there will work best.

The paper tests the next step with **ROME**, a method that changes an MLP weight matrix to update an association. Continue with the authors’ [tracing notebook](https://github.com/kmeng01/rome/blob/main/notebooks/causal_trace.ipynb) to average these traces over many facts, as the paper’s figures do, then their [editing notebook](https://github.com/kmeng01/rome/blob/main/notebooks/rome.ipynb).

An edit should work on the requested prompt, transfer to paraphrases, and preserve neighboring facts. Those three checks connect an impressive single example to the paper’s broader result.

**Paper vs code.** ROME’s update uses a *key*: the MLP input that represents the subject. The paper averages this key over copies of the prompt with several random prefixes. Gupta et al., [“Rebuilding ROME”](https://arxiv.org/abs/2403.07175) (EMNLP 2024), found that the released code uses the averaged key in one part of the update and the key from the bare prompt in another. Most single edits still work, but some facts produce “disabling edits”: unusually large updates that make the model collapse. In a long sequence of edits one of these is eventually hit, which ends sequential editing. Using the averaged key throughout (their r-ROME) removes the collapse.


# 3️⃣ Choose a follow-up

Once your replication checks pass, change one assumption and predict the result before running it.

- **Does feature-selection order matter?** Select SAE features separately inside each validation fold. Does the comparison change?
- **Which baseline matters most?** Remove one baseline at a time. Does the apparent benefit of SAEs depend on the alternatives available?
- **Does tracing survive different corruption?** Vary noise strength while reusing the same underlying draws. Compare answer probabilities and recovery maps.
- **Does tracing predict a good edit?** Compare restoration effects with edit success on paraphrases and neighboring facts.

Use the workspace below to record one question, the comparison you ran, and what you learned.


In [ ]:
experiment = {
    'replication_checkpoint_completed':'',
    'question':'', 'prediction':'', 'changed_factor':'',
    'held_fixed':'', 'metric_and_control':'',
    'result':'', 'discrepancy_or_limitation':''
}
# Implement the follow-up here. Save code, figures, and the filled-in record.


# Discuss your findings

Show your replicated result, explain any unresolved discrepancy, and present your follow-up. Be explicit about which experiments finished. End with one claim your evidence supports and one it does not.

<details><summary>Setup notes, sources, and validation status</summary>

The notebook uses pinned author code for the SAE comparison. Original data and activation files must be prepared through the authors’ release. Sparse encodings keep the probe-fitting step manageable; preparing the largest SAE may require more memory than a standard Colab runtime. Save training outputs to persistent storage if jobs will continue across sessions.

Sources: [SAE probing paper](https://arxiv.org/abs/2502.16681), [author code](https://github.com/JoshEngels/SAE-Probes/tree/9917d94e8ab568e3686b6d3801fab52cda54b81d), [ROME paper](https://arxiv.org/abs/2202.05262), [ROME code](https://github.com/kmeng01/rome). Exercise presentation is inspired by [ARENA](https://www.arena.education/curriculum).

Version 2 was run top to bottom on 2026-10-23 on Python 3.13 with the install cell above, on CPU only (no GPU was free): Checkpoint A with pandas 2.2.3 and 3.1.0, and the GPT-2 XL half with LOW_RAM off (float32) and on (float16). The Gemma Scope JumpReLU encoder matched sae-lens 5.10.6 exactly on the 16k SAE. Full original-artifact retraining (Checkpoint B) and SAE encoding of the original activations have not been run as part of authoring. If an exercise raises `NotImplementedError`, finish it or enable its reference solution. For import problems, restart after installing in a fresh runtime.

</details>
